# TinyCeNN-LM — Story v2: Memory + Low-Rank Story Head

This fast Colab upgrades the Transformer-free Story-AntiRepeat checkpoint to improve **entity consistency, prompt adherence, and plot continuity** without a slow evaluation pass.

Story-v2 adds:
- causal global prefix memory (`192 → 32 → 192`);
- rank-4 low-rank LM-head adapter;
- continuation-oriented TinyStories training;
- prompt labels masked so character/object reuse is not punished as repetition;
- lighter anti-repetition loss;
- <650K trainable-parameter guard and 45-minute hard cap.


In [ ]:
import subprocess, sys, pathlib, importlib
subprocess.run(['nvidia-smi'], check=False)
REPO_DIR = pathlib.Path('/content/TinyCeNN-LM')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/vtavakkoli/TinyCeNN-LM.git', str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR), 'huggingface_hub'], check=True)
SRC = REPO_DIR / 'src'
if str(SRC) not in sys.path: sys.path.insert(0, str(SRC))
importlib.invalidate_caches()
import tinycenn_lm
print('TinyCeNN import: PASS', tinycenn_lm.__file__)


## Hugging Face login
Add a write token to Colab Secrets as `HF_TOKEN`.


In [ ]:
from google.colab import userdata
from huggingface_hub import HfApi, login, snapshot_download
HF_TOKEN = userdata.get('HF_TOKEN')
if not HF_TOKEN:
    raise RuntimeError('Add HF_TOKEN to Colab Secrets first.')
login(token=HF_TOKEN, add_to_git_credential=False)
api = HfApi(token=HF_TOKEN)
HF_USER = api.whoami()['name']
print('HF user:', HF_USER)


## Settings
No held-out benchmark is run. Training stops at 15M continuation tokens or 45 minutes.


In [ ]:
SOURCE_REPO = 'vtava/TinyCeNN-LM-Story-AntiRepeat'
TARGET_REPO = f'{HF_USER}/TinyCeNN-LM-Story-v2'
MAX_TOKENS = 15_000_000
MAX_RUNTIME_MINUTES = 45
LEARNING_RATE = 7e-5
REPEAT_WEIGHT = 0.08
OUTPUT_DIR = pathlib.Path('/content/TinyCeNN-LM/checkpoints/story-v2')
print('source:', SOURCE_REPO)
print('target:', TARGET_REPO)


## Download Story-AntiRepeat warm start


In [ ]:
SOURCE_DIR = pathlib.Path(snapshot_download(SOURCE_REPO, token=HF_TOKEN))
required = ['sharded_moe_cenn_student.pt', 'sharded_moe_student_config.json']
for name in required:
    if not (SOURCE_DIR / name).exists():
        raise FileNotFoundError(f'{SOURCE_REPO} is missing {name}')
print('source checkpoint:', SOURCE_DIR)


## Train Story-v2 — no slow evaluation


In [ ]:
cmd = [
    sys.executable, str(REPO_DIR / 'scripts/train_story_v2.py'),
    '--source-dir', str(SOURCE_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--max-tokens', str(MAX_TOKENS),
    '--max-runtime-minutes', str(MAX_RUNTIME_MINUTES),
    '--learning-rate', str(LEARNING_RATE),
    '--repeat-weight', str(REPEAT_WEIGHT),
    '--memory-rank', '32',
    '--head-rank', '4',
    '--max-trainable-params', '650000',
]
print(' '.join(cmd))
subprocess.run(cmd, cwd=REPO_DIR, check=True)


## Training report


In [ ]:
import json
report = json.loads((OUTPUT_DIR / 'story_v2_training_report.json').read_text())
print(json.dumps(report, indent=2))
assert report['parameters']['trainable'] <= 650_000
assert report['evaluation_performed'] is False


## Publish Story-v2


In [ ]:
readme = '''---
license: mit
pipeline_tag: text-generation
---
# TinyCeNN-LM Story v2
Transformer-free TinyCeNN short-story model with compact 8-shard Top-2 routed FFN, causal global story memory, rank-4 LM-head adapter, continuation training, and anti-repetition loss.

Load with the code in https://github.com/vtavakkoli/TinyCeNN-LM.
'''
(OUTPUT_DIR / 'README.md').write_text(readme)
api.create_repo(TARGET_REPO, repo_type='model', exist_ok=True)
api.upload_folder(repo_id=TARGET_REPO, repo_type='model', folder_path=str(OUTPUT_DIR), commit_message='Publish TinyCeNN Story v2')
print('Published:', f'https://huggingface.co/{TARGET_REPO}')


## Reload published model and test stories
This is only an immediate generation test; it does not run a held-out dataset benchmark.


In [ ]:
import torch
from transformers import AutoTokenizer
from tinycenn_lm.story import repeated_ngram_fraction
from tinycenn_lm.story_v2 import StoryV2ReplacementLayer, LowRankLMHeadAdapter, build_story_v2_student

REMOTE_DIR = pathlib.Path(snapshot_download(TARGET_REPO, token=HF_TOKEN, force_download=True))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dtype = torch.bfloat16 if device.type == 'cuda' and torch.cuda.is_bf16_supported() else (torch.float16 if device.type == 'cuda' else torch.float32)
tok = AutoTokenizer.from_pretrained(REMOTE_DIR, use_fast=True)
if tok.pad_token_id is None: tok.pad_token = tok.eos_token
model = build_story_v2_student(REMOTE_DIR, device=device, dtype=dtype).eval()
layers = model.model.layers if hasattr(model, 'model') else []
assert any(isinstance(m, StoryV2ReplacementLayer) for m in model.modules())
assert any(isinstance(m, LowRankLMHeadAdapter) for m in model.modules())
assert not any(hasattr(m, 'self_attn') or hasattr(m, 'mlp') for m in layers)
print('Transformer-free Story-v2 reload: PASS')

openings = [
    'Mia found a small robot under a tree in the park.',
    'A little fox wanted to reach the top of a snowy mountain.',
    'Tom had a red ball, but one rainy morning it disappeared from the garden.',
]
gen = dict(
    max_new_tokens=110, min_new_tokens=45, do_sample=True,
    temperature=0.68, top_p=0.84, top_k=30,
    repetition_penalty=1.10, no_repeat_ngram_size=4,
    renormalize_logits=True, use_cache=False,
    eos_token_id=tok.eos_token_id, pad_token_id=tok.pad_token_id,
)
for opening in openings:
    prompt = f'Story beginning:\n{opening}\nContinue the story:\n'
    ids = tok(prompt, return_tensors='pt').to(device)
    with torch.inference_mode():
        out = model.generate(**ids, **gen)
    text = tok.decode(out[0], skip_special_tokens=True)
    continuation = text.split('Continue the story:', 1)[-1].strip()
    print('\n' + '=' * 80)
    print('OPENING:', opening)
    print('CONTINUATION:', continuation)
    print(f'repeated 3-gram fraction: {100 * repeated_ngram_fraction(continuation, 3):.2f}%')
